In [ ]:
from pyspark.sql import SparkSession
from google.cloud import storage
import json

In [ ]:
'''
!mkdir -p "$HOME/spark-jars"

!curl -fL \
  "https://storage.googleapis.com/hadoop-lib/gcs/gcs-connector-hadoop3-latest.jar" \
  -o "$HOME/spark-jars/gcs-connector-hadoop3-latest.jar"
'''

In [ ]:
#!ls -lh "$HOME/spark-jars/gcs-connector-hadoop3-latest.jar"

In [ ]:
%pip install delta-spark==4.0.0

In [ ]:
import os
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

gcs_connector_jar = os.path.expanduser(
    "~/spark-jars/gcs-connector-hadoop3-latest.jar"
)

if not os.path.isfile(gcs_connector_jar):
    raise FileNotFoundError(
        f"Nie znaleziono konektora: {gcs_connector_jar}"
    )

builder = (
    SparkSession.builder
    .master("local[*]")
    .appName("development_for_silver_layer")

    # GCS
    .config(
        "spark.jars",
        gcs_connector_jar
    )
    .config(
        "spark.hadoop.fs.gs.impl",
        "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem"
    )
    .config(
        "spark.hadoop.fs.AbstractFileSystem.gs.impl",
        "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS"
    )

    # Delta Lake
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension"
    )
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )
)

spark = (
    configure_spark_with_delta_pip(builder)
    .getOrCreate()
)

In [ ]:
print(spark.version)
print(spark.sparkContext.master)
print(spark.conf.get("spark.sql.extensions", "BRAK"))
print(spark.conf.get("spark.sql.catalog.spark_catalog", "BRAK"))

In [ ]:
'''
spark = SparkSession \
        .builder\
        .appName("development_for_silver_layer") \
        .getOrCreate()
'''

In [ ]:
spark.version

In [ ]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 50)
spark.conf.set("spark.sql.repl.eagerEval.truncate", 100)

In [ ]:
spark.conf.get(
    "spark.sql.adaptive.advisoryPartitionSizeInBytes"
)

In [ ]:
storage_client = storage.Client()
bucket = storage_client.get_bucket("project-dev-storage")

In [ ]:
fil = json.loads(data_string)

#df = spark.createDataFrame(fil)
#df

In [ ]:
print(
    spark.sparkContext
    ._jsc
    .hadoopConfiguration()
    .get("fs.gs.impl")
)

In [ ]:
#######################################  1. SEC ################################

In [ ]:
for i in range(2):
    print(i)

In [ ]:
from pyspark.sql import functions as F

In [ ]:
df_cik = (
        spark.read
        .format("json")
        .load("gs://project-dev-storage/CIK/sec_metadata_2026-08-04 19:51:03.097732+02:00")
)


df_cik_new = df_cik.select("data")#.show()


# !!! fields nie jest strukturą StructType tylko ArrayType 
df_cik = df_cik.select(
            F.explode("data").alias("x")).select(F.col("x")[0].alias("cik"), F.col("x")[1].alias("name"), F.col("x")[2].alias("ticker"), F.col("x")[3].alias("exchange"))#.show()
            
#df_cik.select(F.explode(F.arrays_zip(F.col("fields"), F.col("data"))).alias("x")).select("x.cik").show()

In [ ]:
df_cik.show()

In [ ]:
# ADDING 0 TO LEFT SIZE TO HAVE 10 DIGITS
df_cik = df_cik.withColumn("cik", F.lpad(F.col("cik").cast("string"), 10, "0"))

In [ ]:
df_cik.select("name").show(truncate  = False)

In [ ]:
# defined companies cik list -> for example 5 companies. CIK doesnt change, name can
company_list_cik = ["0001045810", "0000320193", "0001652044", "0000789019", "0001018724"]

In [ ]:
df_cik = df_cik.filter(F.col("cik").isin(company_list_cik))#.show()

In [ ]:
df_cik.show()

In [ ]:
#tickers_list_temp = df_cik.select('ticker').collect()

In [ ]:
#tickers_list_temp

In [ ]:
tickers_list = [row["ticker"] for row in df_cik.select('ticker').collect()]

In [ ]:
tickers_list

In [ ]:
df_cik.limit(20).show()

In [ ]:
df_cik.count()

In [ ]:
''' KOPIA
# ! fields nie jest strukturą StructType tylko ArrayType 
df_cik_new = df_cik.select("data")#.show()

df_cik.select(
            F.explode(F.array(*[F.struct(
               df_cik["data"][i][0].alias("cik"),
               df_cik["data"][i][1].alias("name"),
               df_cik["data"][i][2].alias("ticker"),
               df_cik["data"][i][3].alias("exchange"))
            for i in range(2)])).alias("x")).select("x.cik", "x.name", "x.ticker", "x.exchange").show()
'''

In [ ]:
# df_content.select("facts.us-gaap.*").columns

In [ ]:
df_cik.show()

In [ ]:
######################### 2. SEC - DANE SPÓŁKI ##############################

In [ ]:
df_metadata = (
    spark.read
    .format("json")
    #.option("multiline", "true")
    .load("gs://project-dev-storage/bronze/company_forms/companyForms_2026-08-24 18:29:24.973273+02:00")
)

In [ ]:
df_metadata.limit(20).toPandas()

In [ ]:
from pyspark.sql import functions as F

In [ ]:
df_metadata.select(
    "sic",
    "cik",
    "name",
   # "entityType",
    "sicDescription",
    "fiscalYearEnd",
    F.col("tickers")[0].alias("tickers"),
    F.col("exchanges")[0].alias("exchanges")
    ).show()

In [ ]:
df_test =  df_metadata.select(
        "filings.recent.accessionNumber",
        "filings.recent.filingDate",
        "filings.recent.reportDate",
        "filings.recent.form",
        "filings.recent.primaryDocument",
        "filings.recent.isXBRL",
        "filings.recent.isInlineXBRL"
)

df_test.show()

In [ ]:
df_test

In [ ]:
df_test2 = df_test.select(F.explode(F.arrays_zip(df_test.accessionNumber, df_test.filingDate, df_test.reportDate, df_test.form, df_test.primaryDocument, df_test.isXBRL, df_test.isInlineXBRL)).alias("x")).select(F.col("x.accessionNumber"), F.col("x.filingDate"), F.col("x.reportDate"),
                                                                                                                                                   F.col("x.form"), F.col("x.primaryDocument"), F.col("x.isXBRL"), F.col("x.isInlineXBRL"))
df_test2

In [ ]:
aqe_enabled = spark.conf.get("spark.sql.adaptive.enabled")

print(aqe_enabled)

In [ ]:
#df_test.write.format("delta").mode("overwrite").save("gs://project-dev-storage/company_forms/sec_metadata_folder/2026-08-06/test")
df_test2.hint("REBALANCE").write.mode("overwrite").parquet("gs://project-dev-storage/company_forms/sec_metadata_folder/2026-08-06/test")

In [ ]:
################################################################# XBRL ##################################################################

In [ ]:
# 3 XBRL
df_content = (
    spark.read
    .format("json")
    .load("gs://project-dev-storage/company_forms/sec_facts_folder/2026-08-06/")
)

In [ ]:
#df_temp = spark.read.json("gs://project-dev-storage/company_forms/sec_facts_folder/2026-08-06/0001045810_fact_file_2026-08-06 20:25:53.999746+02:00")
#df_temp.show()

In [ ]:
client = storage.Client()
blob = client.bucket("project-dev-storage").blob("company_forms/sec_facts_folder/2026-08-06/0000320193_fact_file_2026-08-06 20:25:53.999746+02:00")
json_content = blob.download_as_text(encoding="utf-8")

In [ ]:
json_file = json.loads(json_content)
type(json_file)

In [ ]:
json_file["facts"]["us-gaap"]["AcceleratedShareRepurchaseProgramAdjustment"].keys()

In [ ]:
#json_file["filings"]["recent"]["accessionNumber"]#.keys()

In [ ]:
'''
df_content.select(
        "cik",
        "entityName",
        "f
).show()
'''

In [ ]:
#df_content.select(F.explode("facts"))
df_content

In [ ]:
# ZAMIAST AcceleratedShareRepurchaseProgramAdjustment to trzeba by liste po czym ma isc pętla
# "facts.us-gaap.AcceleratedShareRepurchaseProgramAdjustment.units.val"
    # DO WERYFIKACJI CZY zawsze facts.us-gaap czy cos innego ?
df_content2 = df_content.select("cik","entityname", "facts.us-gaap.AcceleratedShareRepurchaseProgramAdjustment.label", F.explode(F.arrays_zip("facts.us-gaap.AcceleratedShareRepurchaseProgramAdjustment.units.usd.val", "facts.us-gaap.AcceleratedShareRepurchaseProgramAdjustment.units.usd.start"))
    .alias("x")).select(F.col("cik"), F.col("entityname"), F.col("label"), F.col("x.val"), F.col("x.start"))
df_content2.show(truncate = False)

In [ ]:
# pobranie nazw kolumn ze schematu
us_gaap_fields = df_content.select("facts.us-gaap.*").columns
#us_gaap_fields = us_gaap_fields[:5]

In [ ]:
us_gaap_fields

In [ ]:
for i in us_gaap_fields:
    print(type(i))

In [ ]:
#df_content_new.select("facts.us-gaap.AcceleratedShareRepurchaseProgramAdjustment.units.usd.val")

In [ ]:
df_content_new = df_content.limit(1)
#df_content_new

In [ ]:
df_content_new.show()

In [ ]:
#df_content_new.schema["facts"].dataType["us-gaap"] # -> StrunctField tych fieldow ktore zawieraja sie w us-gaap

In [ ]:
# LABELS WITH USD ONLY

us_gaap_schema = (
                df_content_new.schema["facts"].dataType["us-gaap"].dataType

)

type(us_gaap_schema)
#us_gaap_schema

In [ ]:
#us_gaap_schema

In [ ]:
us_gaap_fields = df_content.select("facts.us-gaap.*").columns
#us_gaap_fields
#us_gaap_fields = us_gaap_fields[:5]

In [ ]:
##################################################### TESTY ################################

In [ ]:
from pyspark.sql.types import StructType

In [ ]:
#us_gaap_schema

In [ ]:
for i in us_gaap_fields:
    fact_schema = us_gaap_schema[i].dataType
    print(fact_schema)
    #if "units" in fact_schema.fieldNames():
    #    units_schema = fact_schema["units"].dataType
    #    print(units_schema)
    break

In [ ]:
from pyspark.sql.types import ArrayType

In [ ]:
usd_labels = []
for i in us_gaap_fields:
    fact_schema = us_gaap_schema[i].dataType
    if "units" in fact_schema.fieldNames():
        units_schema = fact_schema["units"].dataType
        #print(units_schema)
        if isinstance(units_schema, StructType):
            if "USD" in units_schema.fieldNames():
               units_schema2 = units_schema["USD"].dataType
               #rint(units_schema2)
               if isinstance(units_schema2, ArrayType):
                   usd_element_schema = units_schema2.elementType
                   #print(usd_element_schema)
                   #print(usd_element_schema.fieldNames())
                   if "start" in usd_element_schema.fieldNames():
                       usd_labels.append(i)     
               

''' KOPIA

usd_labels = []
for i in us_gaap_fields:
    fact_schema = us_gaap_schema[i].dataType
    if "units" in fact_schema.fieldNames():
        units_schema = fact_schema["units"].dataType
        #print(units_schema)
        if isinstance(units_schema, StructType):
            if "USD" in units_schema.fieldNames():
               usd_labels.append(i) 

''' 

In [ ]:
len(usd_labels)

In [ ]:
usd_labels

In [ ]:
len(us_gaap_fields)

In [ ]:
#us_gaap_fields = us_gaap_fields[:5]
#us_gaap_fields

In [ ]:
#usd_labels = usd_labels[:5]
#usd_labels

In [ ]:
df_content_new.show()

In [ ]:
for i in usd_labels:
    print(i)

In [ ]:
df_content2 = (df_content_new.select("cik","entityname", F.lit("USD").alias("units"), 
                                                              
                       F.explode(F.array(*[F.struct
                                  (
                                    F.col(f"facts.us-gaap.{i}.label").alias("label"),
                                    F.col(f"facts.us-gaap.{i}.units.USD.start").alias("start"),
                                    F.col(f"facts.us-gaap.{i}.units.USD.end").alias("end"),
                                    F.col(f"facts.us-gaap.{i}.units.USD.val").alias("val"),
                                    F.col(f"facts.us-gaap.{i}.units.USD.accn").alias("accn"),
                                    F.col(f"facts.us-gaap.{i}.units.USD.fy").alias("fy"),
                                    F.col(f"facts.us-gaap.{i}.units.USD.fp").alias("fp"),
                                    F.col(f"facts.us-gaap.{i}.units.USD.form").alias("form"),
                                    F.col(f"facts.us-gaap.{i}.units.USD.filed").alias("filed")
                                  )
                                for i in usd_labels])
                            ).alias("fact")
                
                    ).select("cik","entityname", "units", "fact.label",F.explode(F.arrays_zip("fact.start", "fact.end", "fact.val", "fact.accn", "fact.fy", "fact.fp", "fact.form", "fact.filed")).alias("fact_new"))
    .select("cik","entityname", "units", "label","fact_new.start","fact_new.end", "fact_new.val", "fact_new.accn", "fact_new.fy", "fact_new.fp", "fact_new.form", "fact_new.filed")
                                          
)                          
                              
df_content2.show()



''' KOPIA 
df_content2 = df_content_new.select("cik","entityname", 
                                                              
                       F.explode(F.array(*[F.struct
                                  (
                                    F.col(f"facts.us-gaap.{i}.label").alias("label"),
                                    F.col(f"facts.us-gaap.{i}.units.USD.start").alias("start")
                                  )
                                for i in usd_labels])
                            ).alias("fact")
                
                    ).select("cik","entityname","fact.label","fact.start")
                                          
                           
                              
df_content2.show()
'''

In [ ]:
df_content2.count()

In [ ]:
from datetime import date

In [ ]:
df_content2.write.mode("overwrite").parquet(f"gs://project-dev-storage/company_forms/sec_facts_folder/{date.today()}")
#df_content2.hint("REBALANCE").write.mode("overwrite").parquet("")

In [ ]:
df_content2.limit(20).toPandas()

In [ ]:
'''
df_content2 = df_content_new.select("cik","entityname", 
                                        
                              
                        F.explode(F.array(*[F.struct
                                  (
                                    F.col(f"facts.us-gaap.{i}.label").alias("label"),
                                    F.col(f"facts.us-gaap.{i}.units.USD.val").alias("val")
                                  )
                                for i in usd_labels]
                                )
                            ).alias("fact")      
                        ).select("cik", "entityname","fact.label","fact.val")               
                                      
                           
                              
df_content2.show()
'''

In [ ]:

df_content2 = df_content_new.select("cik","entityname", F.explode(F.array(*[f"facts.us-gaap.{i}.label" for i in usd_labels])).alias("label"),
                                    *[f"facts.us-gaap.{i}.units.USD.val" for i in usd_labels]
                                    #*[f"facts.us-gaap.{i}.units.usd" for i in us_gaap_fields]
                                    #F.arrays_zip(*[f"facts.us-gaap.{i}.units.usd.val" for i in us_gaap_fields])
                              )
                              
df_content2.show()


In [ ]:
#df_content2.columns

In [ ]:
'''
                 KOPIA
df_content2 = df_content_new.select("cik","entityname", F.explode(F.array(*[f"facts.us-gaap.{i}.label" for i in us_gaap_fields])).alias("label"),
                                    *[f"facts.us-gaap.{i}.units.USD.val" for i in usd_labels]
                                    #*[f"facts.us-gaap.{i}.units.usd" for i in us_gaap_fields]
                                    #F.arrays_zip(*[f"facts.us-gaap.{i}.units.usd.val" for i in us_gaap_fields])
                              )
                              
df_content2.show()
'''

In [ ]:
################################################################   ALPHA VANTAGE ##########################################

In [ ]:
df = (spark.read
      .format("json")
      .load("gs://project-dev-storage/alpha_vantage/2026-08-16/alpha_vantage_2026-08-16 22:06:01.799483+02:00")
     )

In [ ]:
df.show()

In [ ]:
#df.schema["Time Series (Daily)"].dataType
df.schema["Time Series (Daily)"].dataType["2026-03-24"].dataType.fieldNames()
#df.select("Time Series (Daily).2026-03-24")

In [ ]:
df.select("Meta Data.`2. Symbol`").show()

In [ ]:
# df.schema["Time Series (Daily)"].dataType.fieldNames() # -> wyswietlenie dat

In [ ]:
from pyspark.sql.types import StringType

In [ ]:
df.select("Meta Data.`2. Symbol`", F.col("Time Series (Daily).2026-03-24").cast(StringType()), "Time Series (Daily).2026-03-24.`2. high`").show()

In [ ]:
df.schema["Time Series (Daily)"].dataType["2026-03-24"]

In [ ]:
df.select("Meta Data.`2. Symbol`", F.array("Time Series (Daily).2026-03-24")).show()

In [ ]:
dates_list = df.schema["Time Series (Daily)"].dataType.fieldNames()
#dates_list = dates_list[:5]

In [ ]:
for i in dates_list:
    print(i)

In [ ]:
df.show()

In [ ]:
df_alpha_vantage = df.select("Meta Data.`2. Symbol`", F.explode(F.array(*[
                                                    F.struct(
                                                       F.lit(f"{i}").alias("date"),
                                                       F.col(f"Time Series (Daily).{i}.`1. open`"),
                                                       F.col(f"Time Series (Daily).{i}.`2. high`"),
                                                       F.col(f"Time Series (Daily).{i}.`3. low`"),
                                                       F.col(f"Time Series (Daily).{i}.`4. close`"),
                                                       F.col(f"Time Series (Daily).{i}.`5. volume`")
                                                    )
                                                       for i in dates_list])).alias("x")
         ).select(F.col("`2. Symbol`").alias("symbol"), F.col("x.date").alias("date"), F.col("x.`1. open`").alias("open"), F.col("x.`2. high`").alias("high"), F.col("x.`3. low`").alias("low"), F.col("x.`4. close`").alias("close"), F.col("x.`5. volume`").alias("volume"))#.show()

In [ ]:
#df_alpha_vantage.filter(F.col("symbol") == "AAPL").show()

In [ ]:
# SAVE TO DELTA TABLE
df_alpha_vantage.write.format("delta").mode("overwrite").save(f"gs://project-dev-storage/alpha_vantage/2026-08-16/silver/alpha_vantage{date.today()}")

In [ ]:
'''
df.select("Meta Data.`2. Symbol`", "Time Series (Daily).2026-03-24", "Time Series (Daily).2026-03-24.`1. open`", 
          "Time Series (Daily).2026-03-24.`2. high`", "Time Series (Daily).2026-03-24.`3. low`", 
          "Time Series (Daily).2026-03-24.`4. close`", "Time Series (Daily).2026-03-24.`5. volume`").show()
'''

In [ ]:
df.select("Meta Data.`2. Symbol`", F.explode(F.array(*[F.lit(f"{i}") for i in dates_list]))).show()

In [ ]:
''' KOPIA
df.select("Meta Data.`2. Symbol`", "Time Series (Daily).2026-03-24", "Time Series (Daily).2026-03-24.`1. open`", 
          "Time Series (Daily).2026-03-24.`2. high`", "Time Series (Daily).2026-03-24.`3. low`", 
          "Time Series (Daily).2026-03-24.`4. close`", "Time Series (Daily).2026-03-24.`5. volume`").show()
'''

In [ ]:
############################################## SILVER - FINAL #####################################

In [ ]:
from pyspark.sql import SparkSession
from google.cloud import storage
import json
from pyspark.sql import functions as F
import os
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
import datetime
from zoneinfo import ZoneInfo

In [ ]:
from pyspark.sql.types import (
            StructType,
            StructField,
            StringType,
            IntegerType,
            DecimalType,
            ArrayType,
            DataType,
            LongType,
            MapType
)

In [ ]:
gcs_connector_jar = os.path.expanduser(
    "~/spark-jars/gcs-connector-hadoop3-latest.jar"
)

if not os.path.isfile(gcs_connector_jar):
    raise FileNotFoundError(
        f"Nie znaleziono konektora: {gcs_connector_jar}"
    )

builder = (
    SparkSession.builder
    .master("local[*]")
    .appName("development_for_silver_layer")

    # GCS
    .config(
        "spark.jars",
        gcs_connector_jar
    )
    .config(
        "spark.hadoop.fs.gs.impl",
        "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem"
    )
    .config(
        "spark.hadoop.fs.AbstractFileSystem.gs.impl",
        "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS"
    )

    # Delta Lake
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension"
    )
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )
)

spark = (
    configure_spark_with_delta_pip(builder)
    .getOrCreate()
)

In [ ]:
storage_client = storage.Client()
bucket = storage_client.get_bucket("project-dev-storage")

In [ ]:
###### 1.1. CompanyTickers - Extract - not necessary

In [ ]:
df_cik = (
        spark.read
        .format("json")
        .load("gs://project-dev-storage/CIK/sec_metadata_2026-08-04 19:51:03.097732+02:00")
)

In [ ]:
start_time = datetime.datetime.now(ZoneInfo('Europe/Warsaw'))

In [ ]:
datetime.date.today()

In [ ]:
df_cik = (
        spark.read
        .format("delta")
        .load(f"gs://project-dev-storage/staging/company_tickers/companyTickers_stg_{datetime.date.today()}/")
)

In [ ]:
df_cik.show()

In [ ]:
###### 2.1. CompanyForms - Extract

In [ ]:
df_companyForms = (
    spark.read
    .format("json")
    #.option("multiline", "true")
    .load(f"gs://project-dev-storage/bronze/company_forms/{datetime.date.today()}/")
)

In [ ]:
df_companyForms =  df_companyForms.select(
        "filings.recent.accessionNumber",
        "filings.recent.filingDate",
        "filings.recent.reportDate",
        "filings.recent.form",
        "filings.recent.primaryDocument",
        "filings.recent.isXBRL",
        "filings.recent.isInlineXBRL"
)

df_test.show()


In [ ]:
df_companyForms = df_companyForms.select(F.explode(F.arrays_zip(df_companyForms.accessionNumber, df_companyForms.filingDate, df_companyForms.reportDate, df_companyForms.form, df_companyForms.primaryDocument, df_companyForms.isXBRL, df_companyForms.isInlineXBRL)).alias("x")).select(F.col("x.accessionNumber"), F.col("x.filingDate"), F.col("x.reportDate"),
                                                                                                                                                   F.col("x.form"), F.col("x.primaryDocument"), F.col("x.isXBRL"), F.col("x.isInlineXBRL"))
df_companyForms.show()

In [ ]:
###### 2.2. CompanyForms - Save to Delta Table

In [ ]:
df_companyForms.write.format("delta").mode("overwrite").save(f"gs://project-dev-storage/silver/company_forms/{datetime.date.today()}")

In [ ]:
###### 3.1. CompanyFacts - Extract

In [ ]:
#spark.conf.get("spark.sql.caseSensitive", "true")

In [ ]:
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
timestamp

In [ ]:
### setting schema to laod data

In [ ]:
nested_units_usd = StructType([
    StructField("start", StringType()),
    StructField("end", StringType()),
    StructField("val", LongType()),
    StructField("accn", StringType()),
    StructField("fy", IntegerType()),
    StructField("fp", StringType()),
    StructField("form", StringType()),
    StructField("filed", StringType())
    ]
)

In [ ]:
nested_units_usd

In [ ]:
ArrayType(nested_units_usd)

In [ ]:
nested_units_usd_array = MapType(
    StringType(),
    ArrayType(nested_units_usd)
)

nested_units_usd_array

In [ ]:
category_schema = StructType([
    StructField("label", StringType()),
    StructField("description", StringType()),
    StructField("units", nested_units_usd_array)
])

category_schema

In [ ]:
### combining schema to read 

schema_xbrl = StructType([
    StructField("cik", LongType()),
    StructField("entityName", StringType()),
    StructField(
        "facts",
        StructType([
            StructField(
                "us-gaap",
                MapType(
                    StringType(),
                    category_schema
                )
            )
        ])
    )
])

In [ ]:
# 3 XBRL
df_companyFacts = (
    spark.read
    .format("json")
    .option("multiline", "true")
    .schema(schema_xbrl)
    .load(f"gs://project-dev-storage/bronze/company_facts/2026-08-27/")
)

In [ ]:
df_companyFacts.show()

In [ ]:
#df_companyFacts.select("cik", "entityName", F.explode("facts.`us-gaap`").alias("key", "value")).select("cik", "entityName", "key", F.explode("value")).show()
df_company_facts = (df_companyFacts.select("cik", "entityName", F.explode("facts.`us-gaap`").alias("fact_name", "fact_details")).select("cik","entityName","fact_name",
                                            F.explode("fact_details.units").alias("unit_name","unit_values")).select("cik","entityName", "fact_name","unit_name",F.explode("unit_values").alias("fact_value"))
    .select("cik","entityName", "fact_name","unit_name", "fact_value.start", "fact_value.end", "fact_value.val", "fact_value.accn", "fact_value.fy", "fact_value.fp", "fact_value.form", "fact_value.filed"))

df_company_facts.show()

In [ ]:
###### 3.2. CompanyFacts - Save to Delta Table

In [ ]:
datetime.date.today()

In [ ]:
df_company_facts.write.format("delta").mode("overwrite").save(f"gs://project-dev-storage/silver/company_facts/{datetime.date.today()}/")

In [ ]:
###### 4.1. Alpha Vantage - Load from bronze

In [ ]:
df_alpha_vantage = (spark.read
                    .format("json")
                    .option("multiline",True)
                    .load(f"gs://project-dev-storage/bronze/alpha_vantage/{datetime.date.today()}/")
)

In [ ]:
df_alpha_vantage.show()

In [ ]:
dates_list = df_alpha_vantage.schema["Time Series (Daily)"].dataType.fieldNames()

In [ ]:
#dates_list

In [ ]:
df_alpha_vantage = df_alpha_vantage.select("Meta Data.`2. Symbol`", F.explode(F.array(*[
                                                    F.struct(
                                                       F.lit(f"{i}").alias("date"),
                                                       F.col(f"Time Series (Daily).{i}.`1. open`"),
                                                       F.col(f"Time Series (Daily).{i}.`2. high`"),
                                                       F.col(f"Time Series (Daily).{i}.`3. low`"),
                                                       F.col(f"Time Series (Daily).{i}.`4. close`"),
                                                       F.col(f"Time Series (Daily).{i}.`5. volume`")
                                                    )
                                                       for i in dates_list])).alias("x")
         ).select(F.col("`2. Symbol`").alias("symbol"), F.col("x.date").alias("date"), F.col("x.`1. open`").alias("open"), F.col("x.`2. high`").alias("high"), F.col("x.`3. low`").alias("low"), F.col("x.`4. close`").alias("close"), F.col("x.`5. volume`").alias("volume"))#.show()

In [ ]:
df_alpha_vantage.show()

In [ ]:
df_alpha_vantage.write.format("delta").mode("overwrite").save(f"gs://project-dev-storage/silver/alpha_vantage/{datetime.date.today()}/")